# Claude Computer Use: The Screenshot and Action Loop
Learn Claude's current computer toolset through an offline wire-format exercise and an optional isolated Python demo.

**Claude section, lecture 3.** `RUN_DEMO=False` runs without an API key, launching a browser or controlling a desktop. The offline exercise is a labelled simulation, not a live Claude result. Enable the optional demo only when you intend to run the API against a fresh, network-blocked demonstration browser.

Current Claude API models use `computer_toolset_20260801` with member names such as `screenshot`, `left_click` and `type`. Calls and results carry `toolset_name="computer"`. Older model integrations may require `computer_20251124` and a beta header; match the tool version to your selected model. This notebook targets the current toolset on Sonnet 5.5.

[Current computer use guide](https://platform.claude.com/docs/en/agents-and-tools/tool-use/computer-use-tool) · [Result formatting](https://platform.claude.com/docs/en/agents-and-tools/tool-use/handle-tool-calls)

The model chooses actions. Your application supplies the environment and runs them. For tasks entirely inside a webpage, consider Claude's browser toolset or a direct application API as well; choose the simplest suitable tool.

In [ ]:
%pip install anthropic playwright pandas --upgrade --quiet

In [ ]:
RUN_DEMO = False
MODEL = "claude-sonnet-5-5"
MAX_TURNS = 12
MAX_ACTIONS_PER_TURN = 8

In [ ]:
import os, json, asyncio, base64, platform, subprocess
from getpass import getpass
import anthropic
from IPython.display import display, Image
if type(RUN_DEMO) is not bool:
    raise TypeError("RUN_DEMO must be True or False, not text")
if RUN_DEMO and not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

## The current request shape
`configs` withholds members we do not implement. The toolset uses screenshot-pixel coordinates; do not attach the old `display_width_px` or `display_height_px` fields. A computer action is identified by both `toolset_name` and the member `name`. A custom tool could otherwise share the same name.

In [ ]:
ALL_MEMBERS = {"screenshot", "zoom", "left_click", "right_click", "middle_click", "double_click", "triple_click",
    "left_click_drag", "mouse_move", "left_mouse_down", "left_mouse_up", "cursor_position", "scroll", "type", "key", "hold_key", "wait"}
ENABLED_MEMBERS = {"screenshot", "left_click", "mouse_move", "cursor_position", "type", "key", "scroll", "wait"}
COMPUTER_TOOL = {"type": "computer_toolset_20260801",
    "configs": {name: {"enabled": False} for name in sorted(ALL_MEMBERS - ENABLED_MEMBERS)}}
request_example = {"model": MODEL, "max_tokens": 1200, "tools": [COMPUTER_TOOL],
    "messages": [{"role": "user", "content": "Take a screenshot, then fill the demo form. Use only the supplied computer tools."}]}
display(request_example)

## Offline exercise: preserve batch order and matching IDs
These are simulated `tool_use` blocks. No API or browser is involved. A real screenshot result contains image blocks; our fixture only demonstrates the click/type result envelopes. Member results must echo `toolset_name`, and all results immediately follow the assistant tool-use message.

In [ ]:
fixture_calls = [
    {"type": "tool_use", "id": "fixture-click", "toolset_name": "computer", "name": "left_click", "input": {"coordinate": [140,260]}},
    {"type": "tool_use", "id": "fixture-type", "toolset_name": "computer", "name": "type", "input": {"text": "Ada Lovelace"}},
]
fixture_log = []
fixture_results = []
for call in fixture_calls:
    assert call["toolset_name"] == "computer" and call["name"] in ENABLED_MEMBERS
    fixture_log.append((call["name"],call["input"]))
    fixture_results.append({"type": "tool_result", "tool_use_id": call["id"], "toolset_name": "computer",
        "content": [{"type": "text", "text": "SIMULATED: action recorded, not executed"}]})
fixture_history = [{"role": "assistant", "content": fixture_calls}, {"role": "user", "content": fixture_results}]
assert [r["tool_use_id"] for r in fixture_results] == [c["id"] for c in fixture_calls]
assert [name for name,args in fixture_log] == ["left_click", "type"]
display(fixture_history)

## Optional: prepare the isolated demonstration browser
The supplied form is local HTML. It has no login, no connection to a real website and no submission endpoint. We use a browser viewport as a small controlled screen; this is not a complete operating-system desktop implementation. Disabled tool members reflect that limitation. Setup downloads Chromium only when `RUN_DEMO=True`.

In [ ]:
if RUN_DEMO:
    install = [os.sys.executable, "-m", "playwright", "install"]
    if platform.system() == "Linux":
        install.append("--with-deps")
    subprocess.run(install + ["chromium"], check=True)

In [ ]:
DEMO_HTML = '\n<!doctype html><html><head><title>Course contact form</title>\n<style>\nbody {font:22px Arial; padding:55px; background:#f5f6fa; color:#242424}\nmain {background:white; padding:35px; width:560px}\nlabel {display:block; margin:20px 0 8px}\ninput {font:22px Arial; padding:12px; width:490px}\nbutton {font:22px Arial; padding:14px 22px; margin-top:25px}\n#preview {margin-top:25px; color:#00695c}\n</style></head><body><main><h1>Course contact form</h1>\n<p>Demo only. Nothing is sent or saved online.</p>\n<label for="name">Name</label><input id="name" autocomplete="off">\n<label for="email">Email</label><input id="email" autocomplete="off">\n<button id="preview-button" onclick="\ndocument.querySelector(\'#preview\').textContent =\n\'Preview: \' + document.querySelector(\'#name\').value + \' | \' + document.querySelector(\'#email\').value;\n">Preview</button><p id="preview"></p></main></body></html>\n'

## The Python executor
Only the enabled members are implemented. Coordinates, key names, text size and wait duration are checked. Unsupported calls become error results. Actions run in the returned order. The executor does not evaluate model-generated code or control your existing browser.

Screenshot contents can contain misleading instructions. Keep the user task separate, isolate the environment and require confirmation for real-world actions when designing your own product.

In [ ]:
KEYS = {"Return": "Enter", "Enter": "Enter", "Tab": "Tab", "Escape": "Escape", "BackSpace": "Backspace", "Backspace": "Backspace", "Delete": "Delete", "space": "Space", "Space": "Space", "ctrl": "Control", "Control": "Control", "shift": "Shift", "Shift": "Shift", "alt": "Alt", "Alt": "Alt", "super": "Meta", "Meta": "Meta", "a": "a", "Home": "Home", "End": "End", "Left": "ArrowLeft", "Right": "ArrowRight", "Up": "ArrowUp", "Down": "ArrowDown"}

def validate_member(name,args):
    if name not in ENABLED_MEMBERS or not isinstance(args,dict):
        raise ValueError("Unknown or disabled computer member")
    allowed={"screenshot":set(), "left_click":{"coordinate","text"}, "mouse_move":{"coordinate"},
             "cursor_position":set(), "type":{"text"}, "key":{"text","repeat"},
             "scroll":{"coordinate","text","scroll_direction","scroll_amount"}, "wait":{"duration"}}[name]
    if set(args)-allowed:
        raise ValueError("Unexpected member arguments")
    if name in {"type","key"} and not isinstance(args.get("text"),str):
        raise ValueError("Text is required")
    if "coordinate" in args:
        xy=args["coordinate"]
        if not isinstance(xy,list) or len(xy)!=2 or not all(type(v) is int for v in xy) or not (0<=xy[0]<1024 and 0<=xy[1]<768):
            raise ValueError("Coordinate outside the demo screen")
    if name=="mouse_move" and "coordinate" not in args:
        raise ValueError("Coordinate required")
    if name=="type" and len(args["text"])>500:
        raise ValueError("Text too long")
    if name=="key":
        if any(key not in KEYS for key in args["text"].split("+")):
            raise ValueError("Unsupported key combination")
        if type(args.get("repeat",1)) is not int or not 1<=args.get("repeat",1)<=10:
            raise ValueError("Invalid repeat count")
    if name=="wait" and (type(args.get("duration")) not in (int,float) or not 0<=args["duration"]<=2):
        raise ValueError("Wait must be between zero and two seconds")
    if name=="scroll" and (args.get("scroll_direction") not in {"up","down","left","right"} or type(args.get("scroll_amount")) is not int or not 0<=args["scroll_amount"]<=10):
        raise ValueError("Invalid scroll")
    if name in {"left_click","scroll"} and args.get("text"):
        raise ValueError("Modifier clicks/scrolls are disabled in this demo")

async def execute_member(page,name,args,state):
    validate_member(name,args)
    if name=="screenshot":
        return [{"type":"image","source":{"type":"base64","media_type":"image/png","data":base64.b64encode(await page.screenshot()).decode()}}]
    if "coordinate" in args:
        state["cursor"] = args["coordinate"]
        await page.mouse.move(*state["cursor"])
    if name=="left_click":
        await page.mouse.click(*state["cursor"])
    elif name=="type":
        await page.keyboard.insert_text(args["text"])
    elif name=="key":
        combination="+".join(KEYS[key] for key in args["text"].split("+"))
        for _ in range(args.get("repeat",1)):
            await page.keyboard.press(combination)
    elif name=="scroll":
        amount=args["scroll_amount"]*80
        dx,dy={"up":(0,-amount),"down":(0,amount),"left":(-amount,0),"right":(amount,0)}[args["scroll_direction"]]
        await page.mouse.wheel(dx,dy)
    elif name=="wait":
        await asyncio.sleep(args["duration"])
    elif name=="cursor_position":
        return [{"type":"text","text":f"X={state['cursor'][0]}, Y={state['cursor'][1]}"}]
    if page.url != "about:blank":
        raise RuntimeError("Navigation away from the local form is blocked")
    return [{"type":"text","text":"OK"}]

In [ ]:
# Offline checks: no browser is launched and no action is executed.
validate_member("left_click",{"coordinate":[140,260]})
validate_member("type",{"text":"Ada Lovelace"})
for name,args in [("left_click",{"coordinate":[2048,0]}),("key",{"text":"alt+F4"}),("wait",{"duration":100}), ("zoom",{"region":[0,0,10,10]})]:
    try:
        validate_member(name,args)
    except ValueError:
        pass
    else:
        raise AssertionError("Expected invalid or disabled action to be rejected")
print("Offline action-boundary checks passed")

## Optional: run the screenshot/action loop
This cell is skipped by default. When enabled, it opens only a fresh headless demonstration browser, sends screenshots to Claude and checks the final form values directly. It caps model turns and actions per turn, and closes the browser even if a call fails. Every screenshot/model turn incurs API usage.

Show the action trace during teaching. The model's final text alone does not establish that the form was filled correctly.

In [ ]:
async def run_demo():
    from playwright.async_api import async_playwright
    usage=[]
    trace=[]
    async with anthropic.AsyncAnthropic(timeout=120,max_retries=1) as api_client:
        async with async_playwright() as pw:
            browser=await pw.chromium.launch(headless=True)
            try:
                context=await browser.new_context(viewport={"width":1024,"height":768})
                await context.route("**/*", lambda route: route.abort())
                page=await context.new_page()
                await page.set_content(DEMO_HTML)
                state={"cursor":[0,0]}
                screenshot=await execute_member(page,"screenshot",{},state)
                history=[{"role":"user","content":[{"type":"text","text":"Fill the local form with name Ada Lovelace and email ada@example.test, then click Preview. Nothing is submitted. Use only the enabled computer tools. End each group of actions with a screenshot to check the result."},*screenshot]}]
                for turn in range(MAX_TURNS):
                    response=await api_client.messages.create(model=MODEL,max_tokens=1200,tools=[COMPUTER_TOOL],messages=history)
                    usage.append({"turn":turn+1,**response.usage.model_dump()})
                    history.append({"role":"assistant","content":[b.model_dump(exclude_none=True) for b in response.content]})
                    calls=[b.model_dump(exclude_none=True) for b in response.content if b.type=="tool_use"]
                    if response.stop_reason=="end_turn" and not calls:
                        break
                    if response.stop_reason!="tool_use" or not calls or len(calls)>MAX_ACTIONS_PER_TURN:
                        raise RuntimeError(f"Unexpected stop/action count: {response.stop_reason}")
                    results=[]
                    for call in calls:
                        trace.append({"member":call["name"],"input":call["input"]})
                        try:
                            if call.get("toolset_name")!="computer":
                                raise ValueError("Unexpected toolset")
                            content=await execute_member(page,call["name"],call["input"],state)
                            error=False
                        except (ValueError,RuntimeError) as exc:
                            content=[{"type":"text","text":str(exc)}];error=True
                        results.append({"type":"tool_result","tool_use_id":call["id"],"toolset_name":"computer","content":content,"is_error":error})
                    history.append({"role":"user","content":results})
                else:
                    raise RuntimeError("Computer-use turn limit reached")
                assert await page.locator("#name").input_value()=="Ada Lovelace"
                assert await page.locator("#email").input_value()=="ada@example.test"
                assert await page.locator("#preview").inner_text()=="Preview: Ada Lovelace | ada@example.test"
                display(Image(data=await page.screenshot()))
                return {"trace":trace,"usage":usage,"verified":True}
            finally:
                await browser.close()
if RUN_DEMO:
    demo_result=await run_demo()
    display(demo_result)
else:
    print("Live computer-use demo is off. The offline lesson completed without computer control.")

## Summary
- Current toolset requests use member names and `toolset_name="computer"`, not the older `action` envelope.
- Execute batched actions in order and return every matching tool-use ID immediately afterward.
- Screenshots supply visual state; your Python executor supplies the permitted actions.
- The optional demo uses a limited, isolated viewport rather than your existing desktop.
- Match model and tool versions, bound execution and verify the real end state.

**Provider comparison:** OpenAI's computer result items and Claude's member tool results differ. Reuse the environment and verification checks while adapting the API-specific envelope. Prefer a direct API, browser tool or computer tool according to the task, not the provider's brand.